# E-Commerce Late Delivery Prediction
## 01 — Data Loading & Order-Level Table Construction

### Objective
This notebook loads the Olist e-commerce data from PostgreSQL and builds a machine-learning-ready dataset with one row per order.

### Workflow
- Load all relational tables from PostgreSQL
- Inspect shapes, columns, keys, and duplicates
- Aggregate one-to-many tables before joining
- Merge order, customer, item, payment, review, product, and seller information
- Validate the final order-level dataset
- Save the result for the next stage of the ML pipeline

## 1. Database Connection

Connect to the local PostgreSQL database containing the Olist e-commerce tables.

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine, inspect

load_dotenv()

DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "olist")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
inspector = inspect(engine)
tables = inspector.get_table_names()

## 2. Load Dataset Tables

Load all PostgreSQL tables into pandas DataFrames for inspection and further processing.

In [2]:
dataframes = {}

for table in tables:
    df = pd.read_sql(f'SELECT * FROM "{table}"', engine)
    dataframes[table] = df
    
    print(f"\nTable: {table}")
    print(f"Shape: {df.shape}")
    print("Columns:")
    print(df.columns.tolist())
    print("-" * 50)


Table: olist_customers_dataset
Shape: (99441, 5)
Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
--------------------------------------------------

Table: olist_geolocation_dataset
Shape: (1000163, 5)
Columns:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
--------------------------------------------------

Table: olist_orders_dataset
Shape: (99441, 8)
Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
--------------------------------------------------

Table: olist_order_items_dataset
Shape: (112650, 7)
Columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
--------------------------------------------------

Table: olist_order_payments_dataset
Shape: (10

## 3. Key & Duplicate Inspection

Inspect primary and foreign keys to understand table relationships and identify duplicate records before performing any joins.

This step is important because tables such as order items, payments, and reviews can contain multiple rows per order.

In [3]:
for table in tables:
    df = dataframes[table]
    
    print(f"\n--- {table} ---")
    
    for col in ["order_id", "customer_id", "product_id", "seller_id"]:
        if col in df.columns:
            print(
                f"{col}: rows={len(df)}, "
                f"unique={df[col].nunique()}, "
                f"duplicates={df[col].duplicated().sum()}"
            )


--- olist_customers_dataset ---
customer_id: rows=99441, unique=99441, duplicates=0

--- olist_geolocation_dataset ---

--- olist_orders_dataset ---
order_id: rows=99441, unique=99441, duplicates=0
customer_id: rows=99441, unique=99441, duplicates=0

--- olist_order_items_dataset ---
order_id: rows=112650, unique=98666, duplicates=13984
product_id: rows=112650, unique=32951, duplicates=79699
seller_id: rows=112650, unique=3095, duplicates=109555

--- olist_order_payments_dataset ---
order_id: rows=103886, unique=99440, duplicates=4446

--- olist_order_reviews_dataset ---
order_id: rows=99224, unique=98673, duplicates=551

--- olist_products_dataset ---
product_id: rows=32951, unique=32951, duplicates=0

--- olist_sellers_dataset ---
seller_id: rows=3095, unique=3095, duplicates=0

--- product_category_name_translation ---


## 4. Build the Base Order-Level Dataset

Start with the orders table and enrich it with customer information.

The merge is performed using `customer_id`, while preserving one row per order.

In [4]:
# Read the main tables
orders = dataframes["olist_orders_dataset"]
customers = dataframes["olist_customers_dataset"]
items = dataframes["olist_order_items_dataset"]
products = dataframes["olist_products_dataset"]
sellers = dataframes["olist_sellers_dataset"]
translation = dataframes["product_category_name_translation"]

# Join orders with customers
joined_df = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

print("After orders + customers:", joined_df.shape)

After orders + customers: (99441, 12)


## 5. Aggregate Order Items

The order items table has a one-to-many relationship with orders, since a single order may contain multiple products.

To prevent row duplication during the merge, item-level data is aggregated to the order level using:
- Number of items
- Total product price
- Total freight cost

In [5]:
items_agg = items.groupby("order_id").agg(
    item_count=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum")
).reset_index()

print(items_agg.shape)
print(items_agg["order_id"].duplicated().sum())

(98666, 4)
0


In [6]:
joined_df = joined_df.merge(
    items_agg,
    on="order_id",
    how="left"
)

print("After adding order items:", joined_df.shape)
print("Duplicate order IDs:", joined_df["order_id"].duplicated().sum())

After adding order items: (99441, 15)
Duplicate order IDs: 0


## 6. Aggregate Payment Information

Payment records can contain multiple transactions for the same order.

To maintain one row per order, payment information is aggregated into:
- Number of payment transactions
- Total payment value
- Maximum number of installments

In [7]:
payments = dataframes["olist_order_payments_dataset"]

payments_agg = payments.groupby("order_id").agg(
    payment_count=("payment_sequential", "count"),
    total_payment=("payment_value", "sum"),
    max_installments=("payment_installments", "max")
).reset_index()

print(payments_agg.shape)
print(payments_agg["order_id"].duplicated().sum())

(99440, 4)
0


In [8]:
joined_df = joined_df.merge(
    payments_agg,
    on="order_id",
    how="left"
)

print("After adding payments:", joined_df.shape)
print("Duplicate order IDs:", joined_df["order_id"].duplicated().sum())

After adding payments: (99441, 18)
Duplicate order IDs: 0


## 7. Aggregate Review Information

An order may have more than one review record.

Review data is aggregated to the order level using:
- Number of reviews
- Average review score

This preserves the one-row-per-order structure before merging with the main dataset.

In [9]:
reviews = dataframes["olist_order_reviews_dataset"]

reviews_agg = reviews.groupby("order_id").agg(
    review_count=("review_id", "count"),
    avg_review_score=("review_score", "mean")
).reset_index()

print(reviews_agg.shape)
print("Duplicate order IDs:", reviews_agg["order_id"].duplicated().sum())

(98673, 3)
Duplicate order IDs: 0


In [10]:
joined_df = joined_df.merge(
    reviews_agg,
    on="order_id",
    how="left"
)

print("After adding reviews:", joined_df.shape)
print("Duplicate order IDs:", joined_df["order_id"].duplicated().sum())

After adding reviews: (99441, 20)
Duplicate order IDs: 0


## 8. Product & Seller Features

Enrich order items with product and seller information before aggregating them back to the order level.

For each order, the following features are created:
- Number of unique products
- Number of unique sellers
- Number of unique product categories

This adds product and seller diversity information while preserving one row per order in the final dataset.

In [11]:
products = dataframes["olist_products_dataset"]
sellers = dataframes["olist_sellers_dataset"]
translation = dataframes["product_category_name_translation"]

# Add English category name to products
products_info = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

# Connect order items with product and seller information
items_details = items.merge(
    products_info,
    on="product_id",
    how="left"
).merge(
    sellers,
    on="seller_id",
    how="left"
)

print("Items details shape:", items_details.shape)
print(items_details.columns.tolist())

Items details shape: (112650, 19)
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [12]:
product_seller_agg = items_details.groupby("order_id").agg(
    unique_products=("product_id", "nunique"),
    unique_sellers=("seller_id", "nunique"),
    unique_categories=("product_category_name", "nunique")
).reset_index()

print(product_seller_agg.shape)
print("Duplicate order IDs:", product_seller_agg["order_id"].duplicated().sum())

(98666, 4)
Duplicate order IDs: 0


## 9. Final Dataset Validation

Merge the aggregated product and seller features into the main dataset and perform final integrity checks.

The final dataset is validated to ensure:
- One row represents one order
- No duplicate `order_id` values exist
- The total number of unique orders is preserved

In [13]:
# Add product/seller summary
joined_df = joined_df.merge(
    product_seller_agg,
    on="order_id",
    how="left"
)

# Final checks
print("Final shape:", joined_df.shape)
print("Duplicate order IDs:", joined_df["order_id"].duplicated().sum())
print("Unique orders:", joined_df["order_id"].nunique())

Final shape: (99441, 23)
Duplicate order IDs: 0
Unique orders: 99441


## 10. Save Final Dataset

Save the validated order-level dataset as `ml_table.csv`.

This dataset will be used in the next notebook to create the late-delivery target variable and continue the machine learning pipeline.

In [14]:
joined_df.to_csv("ml_table.csv", index=False)

print("ml_table.csv saved successfully!")

ml_table.csv saved successfully!


## Conclusion

The raw relational e-commerce tables were successfully transformed into a single order-level dataset.

### Final Result
- **99,441 orders**
- **23 features**
- **0 duplicate order IDs**
- One row per order

The resulting `ml_table.csv` is ready for target creation and subsequent machine learning steps.